<a href="https://colab.research.google.com/github/Michele-Maestrini/FusionCore/blob/main/Version%20V0/Notebooks/05a_Test_Set_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FusionCore v0 — Phase 5a: Test Set Preparation

**Notebook:** `05a_Test_Set_Preparation.ipynb`  
**Phase:** 5 of 5 (Part A)  
**Objective:** Load the Official NASA Test Set (Iron Wall Protocol), replay Phase 2
normalisation and Phase 3 feature engineering, apply Phase 4 transfer items.

**Input:** Phase 2 frozen parameters, Phase 3 feature manifest, Phase 4 transformers.  
**Output:** `X_test.parquet`, `X_test_nn.parquet`, `meta_test.parquet`, `y_test.parquet`.

**Iron Wall Protocol:** The Official Test Set and RUL ground truth are loaded for
the first and only time in this notebook.

---

### References

- **C-MAPSS:** Saxena, A. & Goebel, K. (2008). NASA Ames.
- **RUL Cap:** Heimes, F.O. (2008). *PHM.*


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1 — Environment Setup (Run First)
# ══════════════════════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2 — Project Constants, Imports & Load Function
# ══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import gc

DRIVE_ROOT      = Path('/content/drive/MyDrive/PI')
CMAPSS_DIR      = DRIVE_ROOT / 'Datasets' / 'CMAPSS'
OUTPUTS_DIR     = DRIVE_ROOT / 'FusionCore' / 'v0' / 'outputs'
REGIME_DICT_DIR = OUTPUTS_DIR / 'regime_dictionary'

CMAPSS_SUBSETS = ['FD001', 'FD002', 'FD003', 'FD004']
CMAPSS_COLUMNS = [
    'unit_id', 'cycle', 'op1', 'op2', 'op3',
    's1', 's2', 's3', 's4', 's5', 's6', 's7',
    's8', 's9', 's10', 's11', 's12', 's13', 's14',
    's15', 's16', 's17', 's18', 's19', 's20', 's21',
]
SENSOR_COLS = [c for c in CMAPSS_COLUMNS if c.startswith('s')]
OP_COLS     = ['op1', 'op2', 'op3']

RUL_CAP              = 125
RANDOM_STATE         = 42
VARIANCE_THRESHOLD   = 1.0e-5
KINEMATIC_WINDOW     = 5
SAFE_DENOM_THRESHOLD = 1e-6

SINGLE_FAULT_SUBSETS = ['FD001', 'FD002']
DUAL_FAULT_SUBSETS   = ['FD003', 'FD004']
CPR_NUMERATOR   = 's7'
CPR_DENOMINATOR = 's3'   # s3, not s5 — s5 is regime-dead in z-space
E_THERMAL_NUM_A = 's12'
E_THERMAL_NUM_B = 's11'
E_THERMAL_DENOM = 's9'
EGT_SENSOR      = 's4'
FATIGUE_SENSORS     = ['s4', 's9', 's7']
LINKED_PAIRS        = [('s8', 's13'), ('s9', 's14')]
HEALTH_INDEX_SENSORS = ['s4', 's7', 's9']
UNIT_KEY = ['subset_origin', 'unit_id']

np.random.seed(RANDOM_STATE)

def load_cmapss(subset, split='train'):
    filepath = CMAPSS_DIR / f'{split}_{subset}.txt'
    df = pd.read_csv(filepath, sep=r'\s+', header=None, names=CMAPSS_COLUMNS)
    df.dropna(axis=1, how='all', inplace=True)
    return df

print('✔ Constants and load function defined.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3 — Load Phase 2/3/4 Frozen Parameters
# ══════════════════════════════════════════════════════════════════════════════

# Phase 2: regime normalisation parameters.
regime_params = joblib.load(REGIME_DICT_DIR / 'regime_zscore_params.pkl')
print(f'✔ Regime Z-score parameters loaded ({len(regime_params)} subsets).')

kmeans_models = {}
for subset_key in ['FD002', 'FD004']:
    km_path = REGIME_DICT_DIR / f'kmeans_{subset_key}.pkl'
    if km_path.exists():
        kmeans_models[subset_key] = joblib.load(km_path)
print(f'✔ K-Means centroids loaded: {list(kmeans_models.keys())}')

# Phase 4 transfer item artefacts.
t5_transformer = joblib.load(OUTPUTS_DIR / 'phase4_yeo_johnson_transformer.pkl')
t5_method_log  = joblib.load(OUTPUTS_DIR / 'phase4_t5_method_log.pkl')
print(f'✔ T5 transformer loaded: {t5_method_log["method"]}')

nn_feature_names = joblib.load(OUTPUTS_DIR / 'nn_feature_names.pkl')
print(f'✔ NN feature names loaded: {len(nn_feature_names)} features')

# Feature manifest — canonical 91-feature column order.
feature_manifest = pd.read_csv(OUTPUTS_DIR / 'feature_manifest.csv')
feature_names_91 = list(feature_manifest['feature'])
print(f'✔ Feature manifest loaded: {len(feature_names_91)} features')

# Derive T3 clip bounds from training data.
X_train = pd.read_parquet(OUTPUTS_DIR / 'X_train.parquet')
T3_P1  = float(X_train['s9_s14_ratio'].quantile(0.01))
T3_P99 = float(X_train['s9_s14_ratio'].quantile(0.99))
print(f'✔ T3 clip bounds: P1={T3_P1:.4f}, P99={T3_P99:.4f}')

# EGT baseline and active sensor list from training data.
fd00u_train_full = pd.read_parquet(OUTPUTS_DIR / 'fd00u_train_featured.parquet')
EGT_BASELINE = float(fd00u_train_full['s4'].mean())
sensor_variance = fd00u_train_full[SENSOR_COLS].var()
active_mask = sensor_variance > VARIANCE_THRESHOLD
active_sensor_columns = list(sensor_variance[active_mask].index)
N_ACTIVE = len(active_sensor_columns)
print(f'✔ EGT baseline: {EGT_BASELINE:.6f}')
print(f'✔ Active sensors: {N_ACTIVE} ({active_sensor_columns})')

del X_train, fd00u_train_full
gc.collect()

---

## Iron Wall Protocol

**The Official Test Set (`test_FD00x.txt`) and RUL ground truth (`RUL_FD00x.txt`)
are loaded below for the first and only time.** No model retraining is permitted
after seeing these results.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4 — Load Official Test Set & RUL Ground Truth
# ══════════════════════════════════════════════════════════════════════════════

test_raw = {}
rul_true = {}

for subset in CMAPSS_SUBSETS:
    test_raw[subset] = load_cmapss(subset, split='test')
    test_raw[subset]['subset_origin'] = subset

    rul_file = CMAPSS_DIR / f'RUL_{subset}.txt'
    rul_df = pd.read_csv(rul_file, header=None, names=['RUL'])
    rul_df['RUL'] = rul_df['RUL'].clip(upper=RUL_CAP)
    rul_df['unit_id'] = range(1, len(rul_df) + 1)
    rul_df['subset_origin'] = subset
    rul_true[subset] = rul_df

    n_engines = test_raw[subset]['unit_id'].nunique()
    assert n_engines == len(rul_df), (
        f'{subset}: engine count mismatch — test has {n_engines}, RUL has {len(rul_df)}.'
    )
    print(f'{subset}: {n_engines:>4} engines, {len(test_raw[subset]):>6,} rows')

y_test_df = pd.concat(rul_true.values(), ignore_index=True)
print(f'\nTotal test engines: {len(y_test_df)}')
print(f'RUL range: [{y_test_df["RUL"].min()}, {y_test_df["RUL"].max()}]')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 5 — Regime Assignment & Z-Score Normalisation (Phase 2 Replay)
# ══════════════════════════════════════════════════════════════════════════════

test_normed = {}

for subset in CMAPSS_SUBSETS:
    df = test_raw[subset].copy()

    if subset in ['FD001', 'FD003']:
        df['regime_id'] = 0
    else:
        km = kmeans_models[subset]
        df['regime_id'] = km.predict(df[OP_COLS])

    params = regime_params[subset]
    for regime_id, rp in params.items():
        mask = df['regime_id'] == regime_id
        if mask.sum() == 0:
            continue
        mu    = rp['mu']
        sigma = rp['sigma']
        for col in SENSOR_COLS:
            if col in mu.index and col in sigma.index:
                s = sigma[col]
                if s > 0:
                    df.loc[mask, col] = (df.loc[mask, col] - mu[col]) / s
                else:
                    df.loc[mask, col] = 0.0

    test_normed[subset] = df
    print(f'{subset}: {len(df):>6,} rows, {df["regime_id"].nunique()} regimes')

fd00u_test = pd.concat(test_normed.values(), ignore_index=True)
fd00u_test = fd00u_test.sort_values(
    ['subset_origin', 'unit_id', 'cycle']
).reset_index(drop=True)

print(f'\nFD00u test: {len(fd00u_test):,} rows, '
      f'{fd00u_test.groupby(UNIT_KEY).ngroups} engines')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 6 — Feature Engineering (Phase 3 Replay on Test Set)
# ══════════════════════════════════════════════════════════════════════════════

# ── Part A: Gap-Resolution Features ────────────────────────────────────────
regime_dummies = pd.get_dummies(fd00u_test['regime_id'], prefix='regime').astype(int)
for r in range(6):
    col = f'regime_{r}'
    if col not in regime_dummies.columns:
        regime_dummies[col] = 0
regime_dummies = regime_dummies[sorted(regime_dummies.columns)]
fd00u_test = pd.concat([fd00u_test, regime_dummies], axis=1)
print(f'G5: regime one-hot encoded ({regime_dummies.shape[1]} columns)')

fd00u_test['fault_mode_family'] = fd00u_test['subset_origin'].apply(
    lambda x: 0 if x in SINGLE_FAULT_SUBSETS else 1
).astype(int)
print('G6: fault_mode_family added')

for s_phys, s_corr in LINKED_PAIRS:
    ratio_col = f'{s_phys}_{s_corr}_ratio'
    denom = fd00u_test[s_corr]
    safe_denom = denom.where(denom.abs() > SAFE_DENOM_THRESHOLD, np.nan)
    fd00u_test[ratio_col] = (fd00u_test[s_phys] / safe_denom).fillna(0)
print('G7: ratio features added (s8_s13_ratio, s9_s14_ratio)')

fd00u_test['health_index'] = fd00u_test[HEALTH_INDEX_SENSORS].mean(axis=1)
print('G10: health_index added')

# ── Part B: Kinematic Expansion ────────────────────────────────────────────
for col in active_sensor_columns:
    fd00u_test[f'{col}_delta'] = fd00u_test.groupby(UNIT_KEY)[col].diff()
    fd00u_test[f'{col}_rmean'] = fd00u_test.groupby(UNIT_KEY)[col].transform(
        lambda x: x.rolling(window=KINEMATIC_WINDOW, min_periods=1).mean()
    )
    fd00u_test[f'{col}_rstd'] = fd00u_test.groupby(UNIT_KEY)[col].transform(
        lambda x: x.rolling(window=KINEMATIC_WINDOW, min_periods=1).std()
    )
fd00u_test.fillna(0, inplace=True)
print(f'Kinematic expansion: {N_ACTIVE} x 3 = {N_ACTIVE * 3} features')

# ── Part C: Virtual Sensors ───────────────────────────────────────────────
denom = fd00u_test[CPR_DENOMINATOR]
safe_denom = denom.where(denom.abs() > SAFE_DENOM_THRESHOLD, np.nan)
fd00u_test['CPR'] = (fd00u_test[CPR_NUMERATOR] / safe_denom).fillna(0)

numerator = fd00u_test[E_THERMAL_NUM_A] * fd00u_test[E_THERMAL_NUM_B]
denom = fd00u_test[E_THERMAL_DENOM]
safe_denom = denom.where(denom.abs() > SAFE_DENOM_THRESHOLD, np.nan)
fd00u_test['E_thermal'] = (numerator / safe_denom).fillna(0)

fd00u_test['EGT_drift'] = fd00u_test[EGT_SENSOR] - EGT_BASELINE
print('Virtual sensors: CPR, E_thermal, EGT_drift')

# ── Part D: Cumulative Fatigue ─────────────────────────────────────────────
for sensor in FATIGUE_SENSORS:
    fatigue_col = f'{sensor}_cumfatigue'
    positive_z = fd00u_test[sensor].clip(lower=0)
    fd00u_test[fatigue_col] = positive_z.groupby(
        [fd00u_test['subset_origin'], fd00u_test['unit_id']]
    ).cumsum()
print('Cumulative fatigue: s4, s9, s7')

# ── Extract Feature Matrix ─────────────────────────────────────────────────
missing_cols = [c for c in feature_names_91 if c not in fd00u_test.columns]
assert len(missing_cols) == 0, f'Missing columns: {missing_cols}'

X_test = fd00u_test[feature_names_91].copy()
meta_test = fd00u_test[['subset_origin', 'unit_id', 'cycle']].copy()

assert X_test.shape[1] == 91, f'Expected 91 features, got {X_test.shape[1]}'
assert X_test.isna().sum().sum() == 0, 'NaN detected in X_test'
print(f'\n✔ X_test: {X_test.shape}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 7 — Apply Phase 4 Transfer Items
# ══════════════════════════════════════════════════════════════════════════════

# T3: s9_s14_ratio [P1, P99] clip.
X_test['s9_s14_ratio'] = X_test['s9_s14_ratio'].clip(lower=T3_P1, upper=T3_P99)
print(f'T3: s9_s14_ratio clipped to [{T3_P1:.4f}, {T3_P99:.4f}]')

# T5: Apply frozen transformer to CPR and E_thermal for NN models.
T5_COLS = ['CPR', 'E_thermal']
X_test_nn = X_test.copy()
X_test_nn[T5_COLS] = t5_transformer.transform(X_test[T5_COLS])
print(f'T5: {t5_method_log["method"]} applied to CPR & E_thermal')

# T2: Exclude s16 features for NN models.
s16_features = [c for c in X_test_nn.columns if c.startswith('s16')]
X_test_nn = X_test_nn.drop(columns=s16_features, errors='ignore')
print(f'T2: s16 features excluded ({len(s16_features)} columns removed)')

# Verify NN feature alignment.
assert list(X_test_nn.columns) == nn_feature_names, (
    'NN feature column mismatch between test and training.'
)

print(f'\n✔ X_test:    {X_test.shape} (for XGBoost — 91 features)')
print(f'✔ X_test_nn: {X_test_nn.shape} (for DeepAR, TFT, NHITS)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 8 — Persist Test Set Artefacts
# ══════════════════════════════════════════════════════════════════════════════

X_test.to_parquet(OUTPUTS_DIR / 'X_test.parquet')
X_test_nn.to_parquet(OUTPUTS_DIR / 'X_test_nn.parquet')
meta_test.to_parquet(OUTPUTS_DIR / 'meta_test.parquet')
y_test_df.to_parquet(OUTPUTS_DIR / 'y_test.parquet')

print('Phase 5a outputs persisted:')
print(f'  X_test.parquet:    {X_test.shape}')
print(f'  X_test_nn.parquet: {X_test_nn.shape}')
print(f'  meta_test.parquet: {meta_test.shape}')
print(f'  y_test.parquet:    {y_test_df.shape}')
print(f'\n✔ Notebook 05a complete. Proceed to 05b–05e for inference.')